# Práctica RAG - Fase 1: Limpieza y Chunking de Documentos

Este notebook reproduce el paso de los documentos originales (`data/raw/`) a los fragmentos procesados (`data/processed/`) que se utilizan en los benchmarks de embeddings y LLMs.

### Por qué usamos EPUB en lugar de OCR/PDF
Los dos documentos oficiales se descargaron directamente en formato **EPUB**, que internamente es un archivo `.zip` con HTML/XHTML estructurado. Esto permite extraer texto limpio sin ruido de cabeceras, pies de página ni columnas rotas, usando librerías estándar de Python (`zipfile`, `re`, `html.parser` y `json`).

### Estrategia de chunking aplicada a ambos documentos
- Respetar primero la estructura natural de cada documento (artículos/anexos en el BOE; capítulos/secciones en Debian).
- Si una sección cabe en **1.400 caracteres**, se conserva entera en un solo fragmento.
- Si supera los 1.400 caracteres, se divide por párrafos en fragmentos de **1.200 caracteres con 200 caracteres de solapamiento (overlap)**.
- Se añade siempre el título del artículo o capítulo al inicio de cada fragmento para no perder contexto durante la recuperación.

In [1]:
import zipfile
import re
import html
import json
import os
from html.parser import HTMLParser

# Rutas compatibles tanto si se ejecuta desde data/ como desde la raíz del proyecto
RAW_DIR = "raw" if os.path.exists("raw") else "data/raw"
OUT_DIR = "processed" if os.path.exists("raw") else "data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

class TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.pieces = []
        self.block_tags = {"p", "div", "h1", "h2", "h3", "h4", "h5", "h6", "li", "br", "tr"}

    def handle_starttag(self, tag, attrs):
        if tag in self.block_tags:
            self.pieces.append("\n")

    def handle_endtag(self, tag):
        if tag in self.block_tags:
            self.pieces.append("\n")

    def handle_data(self, data):
        self.pieces.append(data)

    def get_text(self):
        text = "".join(self.pieces)
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n\s*\n+", "\n\n", text)
        return text.strip()

def chunk_text(text, max_chars=1400, overlap=200):
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = []
    current_len = 0

    for p in paragraphs:
        p = p.strip()
        if not p:
            continue
        if current_len + len(p) > max_chars and current_chunk:
            chunks.append("\n\n".join(current_chunk))
            if len(current_chunk[-1]) <= overlap:
                current_chunk = [current_chunk[-1], p]
                current_len = len(current_chunk[0]) + len(p)
            else:
                current_chunk = [p]
                current_len = len(p)
        else:
            current_chunk.append(p)
            current_len += len(p)

    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
    return chunks

## 1. Procesamiento del Documento 1: Normativa (BOE Estatuto de los Trabajadores)

In [2]:
def process_boe():
    with zipfile.ZipFile(os.path.join(RAW_DIR, "BOE-A-2015-11430-consolidado.epub"), "r") as z:
        data = z.read("OEBPS/conso-2015-11430.xhtml").decode("utf-8")

    matches = re.findall(r"<p\b[^>]*class=\"([^\"]*)\"[^>]*>(.*?)</p>", data, re.DOTALL)

    sections = []
    current_ref = "Preámbulo"
    current_title = "Preámbulo"
    current_body = []

    for cls, content in matches:
        clean = html.unescape(re.sub(r"<[^>]+>", " ", content)).strip()
        clean = re.sub(r"\s+", " ", clean)
        if not clean:
            continue

        if "articulo" in cls or "anexo" in cls:
            if current_body:
                sections.append((current_ref, current_title, "\n\n".join(current_body)))
                current_body = []
            current_title = clean
            m_ref = re.search(r"(Art[ií]culo\s+\d+[\w\.]*|Disposici[oó]n\s+[\w\s\.]+|Anexo\s+[\w\s\.]*)", clean, re.I)
            current_ref = m_ref.group(1).strip() if m_ref else clean[:30]
        else:
            current_body.append(clean)

    if current_body:
        sections.append((current_ref, current_title, "\n\n".join(current_body)))

    chunks = []
    chunk_idx = 0
    for ref, title, body in sections:
        if len(body) <= 1400:
            full_content = f"[{title}]\n{body}".strip()
            chunks.append({
                "id": f"boe_{chunk_idx:04d}",
                "doc": "BOE-Estatuto-Trabajadores",
                "reference": ref,
                "title": title,
                "text": full_content
            })
            chunk_idx += 1
        else:
            sub_chunks = chunk_text(body, max_chars=1200, overlap=200)
            for i, sc in enumerate(sub_chunks):
                full_content = f"[{title} - Parte {i+1}/{len(sub_chunks)}]\n{sc}".strip()
                chunks.append({
                    "id": f"boe_{chunk_idx:04d}",
                    "doc": "BOE-Estatuto-Trabajadores",
                    "reference": ref,
                    "title": f"{title} (Parte {i+1})",
                    "text": full_content
                })
                chunk_idx += 1

    with open(os.path.join(OUT_DIR, "boe_chunks.json"), "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    return chunks

boe_chunks = process_boe()
print(f"BOE procesado correctamente: {len(boe_chunks)} fragmentos guardados en {OUT_DIR}/boe_chunks.json")

BOE procesado correctamente: 420 fragmentos guardados en processed/boe_chunks.json


## 2. Procesamiento del Documento 2: Técnica (Manual del Administrador de Debian)

In [3]:
def process_debian():
    chunks = []
    chunk_idx = 0

    with zipfile.ZipFile(os.path.join(RAW_DIR, "debian-handbook.epub"), "r") as z:
        files = [f for f in sorted(z.namelist()) if f.startswith("OEBPS/ch") and f.endswith(".html")]
        for f in files:
            raw = z.read(f).decode("utf-8")
            t_m = re.search(r"<title>(.*?)</title>", raw, re.I)
            raw_title = html.unescape(t_m.group(1)).strip() if t_m else f

            ref_m = re.search(r"(\d+\.\d+(?:\.\d+)?|Capítulo\s+\d+)", raw_title)
            ref = ref_m.group(1) if ref_m else raw_title[:20]

            parser = TextExtractor()
            parser.feed(raw)
            body = parser.get_text()

            if body.startswith(raw_title):
                body = body[len(raw_title):].strip()

            if len(body) <= 1400:
                full_content = f"[{raw_title}]\n{body}".strip()
                chunks.append({
                    "id": f"deb_{chunk_idx:04d}",
                    "doc": "Debian-Handbook",
                    "reference": ref,
                    "title": raw_title,
                    "text": full_content
                })
                chunk_idx += 1
            else:
                sub_chunks = chunk_text(body, max_chars=1200, overlap=200)
                for i, sc in enumerate(sub_chunks):
                    full_content = f"[{raw_title} - Parte {i+1}/{len(sub_chunks)}]\n{sc}".strip()
                    chunks.append({
                        "id": f"deb_{chunk_idx:04d}",
                        "doc": "Debian-Handbook",
                        "reference": ref,
                        "title": f"{raw_title} (Parte {i+1})",
                        "text": full_content
                    })
                    chunk_idx += 1

    with open(os.path.join(OUT_DIR, "debian_chunks.json"), "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    return chunks

debian_chunks = process_debian()
print(f"Debian procesado correctamente: {len(debian_chunks)} fragmentos guardados en {OUT_DIR}/debian_chunks.json")

Debian procesado correctamente: 1210 fragmentos guardados en processed/debian_chunks.json


## 3. Resumen de los Documentos Procesados

In [4]:
# Pandas se usa únicamente aquí para mostrar la tabla resumen visual
import pandas as pd
from IPython.display import display

resumen = pd.DataFrame([
    {
        "Documento": "1. Normativa (BOE Estatuto de los Trabajadores)",
        "Archivo Original": "BOE-A-2015-11430-consolidado.epub",
        "Fragmentos": len(boe_chunks),
        "Longitud Media (caracteres)": round(sum(len(c["text"]) for c in boe_chunks) / len(boe_chunks), 1)
    },
    {
        "Documento": "2. Técnica (Debian Handbook)",
        "Archivo Original": "debian-handbook.epub",
        "Fragmentos": len(debian_chunks),
        "Longitud Media (caracteres)": round(sum(len(c["text"]) for c in debian_chunks) / len(debian_chunks), 1)
    }
])
display(resumen.style.format({"Longitud Media (caracteres)": "{:.1f}"}).hide(axis="index"))

Documento,Archivo Original,Fragmentos,Longitud Media (caracteres)
1. Normativa (BOE Estatuto de los Trabajadores),BOE-A-2015-11430-consolidado.epub,420,939.1
2. Técnica (Debian Handbook),debian-handbook.epub,1210,1029.0
